In [ ]:
!pip install geopandas -U
!pip install contextily
!pip install climate_indices
!pip install esda
!pip install spreg
!pip install h3==3.7.6

In [ ]:
import os
import h3

import xarray as xr
import pandas as pd
import geopandas as gpd
import numpy as np

from requests import get
from tqdm.auto import tqdm
from climate_indices import compute, indices, utils
from shapely.geometry import Polygon

### Downloading nClimGrid

In [ ]:
years = [year + 1960 for year in range(30)]
months = [str(month + 1) if month > 8 else '0' + str(month + 1) for month in range(12)]

In [ ]:
os.makedirs('/kaggle/tmp/ncgd/base', exist_ok=True)
for year in tqdm(years, total=len(years)):
    for month in months:
        response = get(f'https://www.ncei.noaa.gov/data/nclimgrid-daily/access/grids/{year}/ncdd-{year}{month}-grd-scaled.nc')
        os.makedirs(f'/kaggle/tmp/ncgd/base/{month}', exist_ok=True)
        with open(f'/kaggle/tmp/ncgd/base/{month}/ncdd-{year}{month}.nc', 'wb') as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
            f.close()

In [ ]:
years = [year + 2007 for year in range(13)]

In [ ]:
os.makedirs('/kaggle/tmp/ncgd/target', exist_ok=True)
for year in tqdm(years, total=len(years)):
    for month in months:
        response = get(f'https://www.ncei.noaa.gov/data/nclimgrid-daily/access/grids/{year}/ncdd-{year}{month}-grd-scaled.nc')
        with open(f'/kaggle/tmp/ncgd/target/ncdd-{year}{month}.nc', 'wb') as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
            f.close()

### Calculating temperature anomalies

In [ ]:
for month in tqdm(months, total=len(months)):
    data = xr.open_mfdataset(f'/kaggle/tmp/ncgd/base/{month}/*.nc', combine='by_coords', parallel=True, chunks={'time': 100})
    data = data['tavg']
    mean = data.mean(dim='time')
    std = data.std(dim='time')
    treshold = 3 * std + mean
    treshold.to_netcdf(f'/kaggle/tmp/ncgd/treshold-{month}.nc')

In [ ]:
temp = pd.DataFrame()
for month in tqdm(months, total=len(months)):
    treshold = xr.open_dataset(f'/kaggle/tmp/ncgd/treshold-{month}.nc')
    treshold = treshold['tavg']
    for year in years:
        print('Year:', year, end='\r')
        data = xr.open_dataset(f'/kaggle/tmp/ncgd/target/ncdd-{year}{month}.nc')
        data = data['tavg']
        data = data.where(data > treshold).dropna(dim=('time'), how='all')
        data = data.stack(stacked_dim=('time', 'lat', 'lon'))
        data = data.to_dataframe()
        data = data.dropna(subset=['tavg'])
        data = data.reset_index(drop=True)
        temp = pd.concat([temp, data])

In [ ]:
temp = temp.reset_index(drop=True)
temp = gpd.GeoDataFrame(temp, geometry=gpd.points_from_xy(temp['lon'], temp['lat']), crs='EPSG:4326')
temp['time'] = pd.to_datetime(temp['time'])
temp = temp[['tavg', 'time', 'geometry']]

In [ ]:
temp

In [ ]:
temp.info()

In [ ]:
os.makedirs('/kaggle/working/data', exist_ok=True)
temp.to_file('/kaggle/working/data/temperature_anomalies.gpkg')

In [ ]:
del temp, data, treshold, mean, std

### Calculating percipitation anomalies

In [ ]:
prcp = xr.open_mfdataset('/kaggle/tmp/ncgd/target/*.nc', combine="by_coords", parallel=True)
prcp['time'] = pd.to_datetime(prcp['time'].values)
prcp = prcp['prcp'].resample(time="1ME").sum(dim="time")
prcp = prcp.load()

In [ ]:
spi = xr.full_like(prcp, np.nan)

for lat in tqdm(range(prcp.shape[1]), total=prcp.shape[1]):
    for lon in range(prcp.shape[2]):
        series = prcp[:, lat, lon].values
        if np.isnan(series).all():
            continue
        else:
            spi_series = indices.spi(
                series, 
                scale=1, 
                distribution=indices.Distribution.gamma,
                data_start_year=2007,
                calibration_year_initial=2007, 
                calibration_year_final=2019,
                periodicity=compute.Periodicity.monthly
            )
            spi[:, lat, lon] = spi_series

In [ ]:
spi_high = spi.where(spi>2, drop=True)
spi_low = spi.where(spi<-2, drop=True)

In [ ]:
spi_high = spi_high.to_dataframe()
spi_high = spi_high.dropna(subset=['prcp'])
spi_high = spi_high.reset_index()

spi_low = spi_low.to_dataframe()
spi_low = spi_low.dropna(subset=['prcp'])
spi_low = spi_low.reset_index()

In [ ]:
spi_high = gpd.GeoDataFrame(spi_high, geometry=gpd.points_from_xy(spi_high['lon'], spi_high['lat']), crs='EPSG:4326')
spi_high['time'] = pd.to_datetime(spi_high['time'])
spi_high = spi_high[['prcp', 'time', 'geometry']]

spi_low = gpd.GeoDataFrame(spi_low, geometry=gpd.points_from_xy(spi_low['lon'], spi_low['lat']), crs='EPSG:4326')
spi_low['time'] = pd.to_datetime(spi_low['time'])
spi_low = spi_low[['prcp', 'time', 'geometry']]

In [ ]:
spi_high.to_file('/kaggle/working/data/spi_high.gpkg')
spi_low.to_file('/kaggle/working/data/spi_low.gpkg')

In [ ]:
del series, spi, spi_high, spi_low, prcp, spi_series

### Download aggregation geometries

In [ ]:
url = 'https://public.opendatasoft.com/api/explore/v2.1/catalog/datasets/us-state-boundaries/exports/geojson'
states = gpd.read_file(url)

In [ ]:
not_main_land = ['Alaska', 'American Samoa', 'Commonwealth of the Northern Mariana Islands', 'Guam', 'Hawaii', 'Puerto Rico', 'United States Virgin Islands']
states = states[~states['name'].isin(not_main_land)]
states = states[['geometry', 'stusab']]
states = states.rename(columns={'stusab': 'state'})

In [ ]:
states.plot()

In [ ]:
states

In [ ]:
states.to_file('/kaggle/working/data/states.gpkg')

In [ ]:
url = 'https://public.opendatasoft.com/api/explore/v2.1/catalog/datasets/us-county-boundaries/exports/geojson'
counties = gpd.read_file(url)

In [ ]:
counties.info()

In [ ]:
counties = counties.sjoin(states[['geometry']], how='inner').drop(columns=['index_right'])
counties = counties[['geometry', 'geoid']]
counties = counties.rename(columns={'geoid': 'county'})
counties = counties.drop_duplicates()

In [ ]:
counties

In [ ]:
counties.plot()

In [ ]:
counties.to_file('/kaggle/working/data/counties.gpkg')

In [ ]:
import h3
from shapely.geometry import Polygon

In [ ]:
usa = states.union_all()
polygons = list(usa.geoms)

In [ ]:
hexagons = set()
for polygon in polygons:
    geojson_poly = polygon.__geo_interface__
    hexes = h3.polyfill(geojson_poly, 3, geo_json_conformant=True)
    hexagons.update(hexes)

In [ ]:
geometries = [
    Polygon(h3.h3_to_geo_boundary(hex_id, geo_json=True))
    for hex_id in hexagons
]

cells = gpd.GeoDataFrame(geometry=geometries, crs="EPSG:4326")
cells['cell'] = range(len(cells))

In [ ]:
cells.plot()

In [ ]:
cells

In [ ]:
cells.to_file('/kaggle/working/data/cells.gpkg')

### The Climate Change Twitter Dataset

In [ ]:
twitter = pd.read_csv('/kaggle/input/the-climate-change-twitter-dataset/The Climate Change Twitter Dataset.csv')

In [ ]:
twitter = twitter[(twitter['lat'].notna())&(twitter['lng'].notna())]
twitter = gpd.GeoDataFrame(twitter, geometry=gpd.points_from_xy(twitter['lng'], twitter['lat']), crs='EPSG:4326')
twitter = twitter[['stance', 'created_at', 'geometry']]
twitter = twitter.sjoin(states).drop(columns=['index_right'])
twitter = twitter.sjoin(counties).drop(columns=['index_right'])
twitter = twitter.sjoin(cells).drop(columns=['index_right'])
twitter['created_at'] = pd.to_datetime(twitter['created_at'])
twitter['created_at'] = twitter['created_at'].dt.tz_localize(None)
twitter = twitter[twitter['created_at'] >= pd.Timestamp('2007-01-01')]

In [ ]:
twitter.info()

In [ ]:
twitter.to_file('/kaggle/working/data/twitter.gpkg')

## Aggregation

In [ ]:
import os
import time

import pandas as pd
import geopandas as gpd

In [ ]:
gpd.__version__

In [ ]:
temp = gpd.read_file('/kaggle/working/data/temperature_anomalies.gpkg')
spi_high = gpd.read_file('/kaggle/working/data/spi_high.gpkg')
spi_low = gpd.read_file('/kaggle/working/data/spi_low.gpkg')

twitter = gpd.read_file('/kaggle/working/data/twitter.gpkg')

states = gpd.read_file('/kaggle/working/data/states.gpkg')
counties = gpd.read_file('/kaggle/working/data/counties.gpkg')
cells = gpd.read_file('/kaggle/working/data/cells.gpkg')

In [ ]:
temp.info()

In [ ]:
spi_high.info()

In [ ]:
spi_low.info()

In [ ]:
twitter.info()

In [ ]:
temp = temp.sjoin(states).drop(columns=['index_right'])
temp = temp.sjoin(counties).drop(columns=['index_right'])
temp = temp.sjoin(cells).drop(columns=['index_right'])

spi_high = spi_high.sjoin(states).drop(columns=['index_right'])
spi_high = spi_high.sjoin(counties).drop(columns=['index_right'])
spi_high = spi_high.sjoin(cells).drop(columns=['index_right'])

spi_low = spi_low.sjoin(states).drop(columns=['index_right'])
spi_low = spi_low.sjoin(counties).drop(columns=['index_right'])
spi_low = spi_low.sjoin(cells).drop(columns=['index_right'])

In [ ]:
temp

In [ ]:
spi_high

In [ ]:
spi_low

In [ ]:
temp.to_file('/kaggle/working/data/temperature_anomalies.gpkg')
spi_high.to_file('/kaggle/working/data/spi_high.gpkg')
spi_low.to_file('/kaggle/working/data/spi_low.gpkg')

In [ ]:
os.makedirs('/kaggle/working/data/annual', exist_ok=True)
os.makedirs('/kaggle/working/data/total', exist_ok=True)

In [ ]:
def aggregate_total(anomalies, anomalies_column, aggregation, name):

    spatial_aggs = ['state', 'county', 'cell']
    special_aggs = ['D', 'ME']

    for spatial_agg in spatial_aggs:
        start = time.perf_counter()
        print(f'Processing {name} - {aggregation} - {spatial_agg}', end='.  ')
        twitter_agg = twitter[['stance', spatial_agg]].groupby([spatial_agg]).count()
        twitter_agg = twitter_agg.rename(columns={'stance': 'total'})
        
        twitter_agg_b = twitter[twitter['stance']=='believer'][['stance', spatial_agg]].groupby([spatial_agg]).count()
        twitter_agg_b = twitter_agg_b.rename(columns={'stance': 'believers'})
        
        twitter_agg_d = twitter[twitter['stance']=='denier'][['stance', spatial_agg]].groupby([spatial_agg]).count()
        twitter_agg_d = twitter_agg_d.rename(columns={'stance': 'deniers'})
        
        twitter_agg_n = twitter[twitter['stance']=='neutral'][['stance', spatial_agg]].groupby([spatial_agg]).count()
        twitter_agg_n = twitter_agg_n.rename(columns={'stance': 'neutrals'})

        twitter_agg = twitter_agg.join(twitter_agg_b)
        twitter_agg = twitter_agg.join(twitter_agg_d)
        twitter_agg = twitter_agg.join(twitter_agg_n)

        twitter_agg['believers_pr'] = twitter_agg['believers'] / twitter_agg['total'] * 100
        twitter_agg['deniers_pr'] = twitter_agg['deniers'] / twitter_agg['total'] * 100
        twitter_agg['neutrals_pr'] = twitter_agg['neutrals'] / twitter_agg['total'] * 100

        if aggregation in special_aggs:
            anomalies_agg = anomalies[[
                anomalies_column, spatial_agg, 'time'
            ]].groupby(
                [pd.Grouper(key='time', freq=aggregation), spatial_agg],
                as_index=False
            ).count()
            anomalies_agg = anomalies_agg[[anomalies_column, spatial_agg]].groupby([spatial_agg]).count()
        else:
            anomalies_agg = anomalies[[anomalies_column, spatial_agg]].groupby([spatial_agg]).agg(aggregation)

        twitter_agg = twitter_agg.reset_index()
        anomalies_agg = anomalies_agg.reset_index()
        total = twitter_agg.merge(anomalies_agg, how='left', on=spatial_agg)
        total = total.fillna(0)
        
        if spatial_agg == 'state':
            total = total.merge(states, on=spatial_agg)
        elif spatial_agg == 'county':
            total = total.merge(counties, on=spatial_agg)
        else:
            total = total.merge(cells, on=spatial_agg)

        total = gpd.GeoDataFrame(total, geometry=total['geometry'], crs='EPSG:4326')
        total = total.rename(columns={anomalies_column: 'anomaly'})
        total = total.drop_duplicates(subset=[spatial_agg])
        total = total[['believers_pr', 'anomaly', 'geometry']]
        total.to_file(f'/kaggle/working/data/total/{name}_{aggregation}_{spatial_agg}.gpkg')
        end = time.perf_counter()
        print(f'Time: {end - start:.2f}s')

In [ ]:
aggregate_total(temp, 'tavg', 'count', 'temperature')
aggregate_total(temp, 'tavg', 'sum', 'temperature')
aggregate_total(temp, 'tavg', 'mean', 'temperature')
aggregate_total(temp, 'tavg', 'D', 'temperature')

aggregate_total(spi_high, 'prcp', 'count', 'spi_high')
aggregate_total(spi_high, 'prcp', 'sum', 'spi_high')
aggregate_total(spi_high, 'prcp', 'mean', 'spi_high')
aggregate_total(spi_high, 'prcp', 'ME', 'spi_high')

aggregate_total(spi_low, 'prcp', 'count', 'spi_low')
aggregate_total(spi_low, 'prcp', 'sum', 'spi_low')
aggregate_total(spi_low, 'prcp', 'mean', 'spi_low')
aggregate_total(spi_low, 'prcp', 'ME', 'spi_low')

In [ ]:
def aggregate_annual(anomalies, anomalies_column, aggregation, name):

    spatial_aggs = ['state', 'county', 'cell']
    special_aggs = ['D', 'ME']

    for spatial_agg in spatial_aggs:
        start = time.perf_counter()
        print(f'Processing {name} - {aggregation} - {spatial_agg}', end='.  ')
        twitter_agg = twitter[['stance', spatial_agg, 'created_at']].groupby(
            [pd.Grouper(key='created_at', freq='YE'), spatial_agg]
        ).count()
        twitter_agg = twitter_agg.rename(columns={'stance': 'total'})
        
        twitter_agg_b = twitter[twitter['stance']=='believer'][['stance', spatial_agg, 'created_at']].groupby(
            [pd.Grouper(key='created_at', freq='YE'), spatial_agg]
        ).count()
        twitter_agg_b = twitter_agg_b.rename(columns={'stance': 'believers'})
        
        twitter_agg_d = twitter[twitter['stance']=='denier'][['stance', spatial_agg, 'created_at']].groupby(
            [pd.Grouper(key='created_at', freq='YE'), spatial_agg]
        ).count()
        twitter_agg_d = twitter_agg_d.rename(columns={'stance': 'deniers'})
        
        twitter_agg_n = twitter[twitter['stance']=='neutral'][['stance', spatial_agg, 'created_at']].groupby(
            [pd.Grouper(key='created_at', freq='YE'), spatial_agg]
        ).count()
        twitter_agg_n = twitter_agg_n.rename(columns={'stance': 'neutrals'})

        twitter_agg = twitter_agg.join(twitter_agg_b)
        twitter_agg = twitter_agg.join(twitter_agg_d)
        twitter_agg = twitter_agg.join(twitter_agg_n)

        twitter_agg['believers_pr'] = twitter_agg['believers'] / twitter_agg['total'] * 100
        twitter_agg['deniers_pr'] = twitter_agg['deniers'] / twitter_agg['total'] * 100
        twitter_agg['neutrals_pr'] = twitter_agg['neutrals'] / twitter_agg['total'] * 100

        twitter_agg =twitter_agg.rename_axis(index={'created_at': 'time'})

        if aggregation in special_aggs:
            anomalies_agg = anomalies[[
                anomalies_column, spatial_agg, 'time'
            ]].groupby(
                [pd.Grouper(key='time', freq=aggregation), spatial_agg],
                as_index=False
            ).count()
            anomalies_agg = anomalies_agg[[anomalies_column, spatial_agg, 'time']].groupby(
                [pd.Grouper(key='time', freq='YE'), spatial_agg]
            ).count()
        else:
            anomalies_agg = anomalies[[anomalies_column, spatial_agg, 'time']].groupby(
                [pd.Grouper(key='time', freq='YE'), spatial_agg]
            ).agg(aggregation)

        twitter_agg = twitter_agg.reset_index()
        anomalies_agg = anomalies_agg.reset_index()
        total = twitter_agg.merge(anomalies_agg, how='left', on=[spatial_agg, 'time'])
        total = total.fillna(0)
        
        total = total.rename(columns={anomalies_column: 'anomaly'})

        total.to_csv(f'/kaggle/working/data/annual/{name}_{aggregation}_{spatial_agg}.csv')
        end = time.perf_counter()
        print(f'Time: {end - start:.2f}s')

In [ ]:
aggregate_annual(temp, 'tavg', 'count', 'temperature')
aggregate_annual(temp, 'tavg', 'sum', 'temperature')
aggregate_annual(temp, 'tavg', 'mean', 'temperature')
aggregate_annual(temp, 'tavg', 'D', 'temperature')

aggregate_annual(spi_high, 'prcp', 'count', 'spi_high')
aggregate_annual(spi_high, 'prcp', 'sum', 'spi_high')
aggregate_annual(spi_high, 'prcp', 'mean', 'spi_high')
aggregate_annual(spi_high, 'prcp', 'ME', 'spi_high')

aggregate_annual(spi_low, 'prcp', 'count', 'spi_low')
aggregate_annual(spi_low, 'prcp', 'sum', 'spi_low')
aggregate_annual(spi_low, 'prcp', 'mean', 'spi_low')
aggregate_annual(spi_low, 'prcp', 'ME', 'spi_low')

## Analysis

### Spatial Analysis

In [ ]:
import os

import pandas as pd
import geopandas as gpd
import contextily as ctx
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe

from spreg import ML_Lag, LMtests, OLS
from esda.moran import Moran, Moran_Local, Moran_BV, Moran_Local_BV
from libpysal.weights.contiguity import Queen
from tqdm.auto import tqdm
from matplotlib.colors import ListedColormap
from matplotlib.colors import to_rgba
from matplotlib.ticker import FuncFormatter
from shapely.affinity import translate

In [ ]:
os.makedirs('/kaggle/working/graphics/spatial', exist_ok=True)

In [ ]:
total_list = os.listdir('/kaggle/working/data/total/')

In [ ]:
def plot_moran(tmp, name):
    categories = {1: 'Горячая точка', 2: 'Холодный выброс', 3: 'Холодная точка', 4: 'Горячий выброс'}
    colors = ['red', 'salmon', 'blue', 'cornflowerblue']
    
    tmp['Q_label'] = tmp['Q'].map(categories)
    cmap = ListedColormap(colors)
    
    tmp_shadow = tmp.copy()
    dx, dy = 0.4, -0.4
    tmp_shadow.geometry = tmp_shadow.geometry.apply(
        lambda geom: translate(geom, xoff=dx, yoff=dy)
    )
    
    fig, ax = plt.subplots(figsize=(32, 18))
    
    tmp_shadow.plot(
        ax=ax,
        color=to_rgba('black', alpha=0.2),
        edgecolor='none',
        zorder=1,
        path_effects=[
            pe.withStroke(linewidth=6, foreground='gray', alpha=0.3),
            pe.Normal()
        ]
    )
    
    tmp.plot(
        'Q_label', 
        legend=True, 
        categorical=True,
        ax=ax,
        cmap=cmap,
        edgecolor='white',
        linewidth=2,
        legend_kwds={
            'loc': 'lower center',
            'bbox_to_anchor': (0.5, -0.15),
            'prop': {'size': 24},
            'ncol': 4,
            'frameon': False
        }
    )
    tmp[tmp['P'] > 0.05].plot(
        color='silver',
        edgecolor='white',
        linewidth=2,
        ax=ax
    )
    
    
    def format_lon(x):
        return f"{abs(x):.0f}°{'W' if x < 0 else 'E'}"
    
    def format_lat(y):
        return f"{abs(y):.0f}°{'S' if y < 0 else 'N'}"

    ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: format_lon(x)))
    ax.yaxis.set_major_formatter(FuncFormatter(lambda y, _: format_lat(y)))
    ax.tick_params(axis='both', labelsize=20)
    
    ax.grid(color='black', linewidth=0.5)
    ctx.add_basemap(ax, source=ctx.providers.CartoDB.PositronNoLabels, crs=tmp.crs)
    ctx.add_basemap(ax, source=ctx.providers.CartoDB.PositronOnlyLabels, zoom=5, crs=tmp.crs)
    
    plt.savefig('/kaggle/working/graphics/spatial/'+name+'.png', dpi=420, bbox_inches='tight')
    plt.close()

In [ ]:
report = pd.DataFrame()
for filename in tqdm(total_list, total=len(total_list)):
    tmp = gpd.read_file(f'/kaggle/working/data/total/{filename}')
    tmp = tmp.dropna()
    name = filename[:-5]
    
    y = tmp['believers_pr'].values
    x = tmp['anomaly'].values
    w = Queen.from_dataframe(tmp, use_index=True, silence_warnings=True)
    w.transform = 'r'

    moran_bv = Moran_BV(y, x, w)
    moran_loc_bv = Moran_Local_BV(y, x, w, island_weight=0)
    
    tmp['Q'] = moran_loc_bv.q
    tmp['P'] = moran_loc_bv.p_sim

    report.loc[name, 'Moran'] = moran_bv.I
    report.loc[name, 'P Moran'] = moran_bv.p_sim
    
    #plot_moran(tmp, name)

In [ ]:
report = report.sort_index()

In [ ]:
shr = report.iloc[:12].copy().reset_index()
slr = report.iloc[12:24].copy().reset_index()
tmr = report.iloc[24:].copy().reset_index()

In [ ]:
shr[['method', 'space_method']] = shr['index'].str.split('_', expand=True, n=3).iloc[:, -2:]
slr[['method', 'space_method']] = slr['index'].str.split('_', expand=True, n=3).iloc[:, -2:]
tmr[['method', 'space_method']] = tmr['index'].str.split('_', expand=True, n=3).iloc[:, -2:]

In [ ]:
method = {'D': 'Количество дней', 'count': 'Количество', 'mean': 'Среднее', 'sum': 'Сумма', 'ME': 'Количество месяцев'}
space_method = {'cell': 'Гексагоны', 'county': 'Округа', 'state': 'Штаты'}

In [ ]:
shr['method'] = shr['method'].map(method)
shr['space_method'] = shr['space_method'].map(space_method)
slr['method'] = slr['method'].map(method)
slr['space_method'] = slr['space_method'].map(space_method)
tmr['method'] = tmr['method'].map(method)
tmr['space_method'] = tmr['space_method'].map(space_method)

In [ ]:
corr_matrix = tmr.pivot(index='method', columns='space_method', values='Moran')
pvalue_matrix = tmr.pivot(index='method', columns='space_method', values='P Moran')

mask = pvalue_matrix > 0.05

plt.figure(figsize=(10, 5))

sns.heatmap(
    corr_matrix, annot=True, fmt='.3f', cmap='bwr', vmin=-0.5, vmax=0.5, mask=mask, 
    cbar_kws={'label': 'Индекс Морана'}, linecolor='white'
)
sns.heatmap(corr_matrix, mask=~mask, cmap=['#f0f0f0'], annot=True, fmt='.3f', cbar=False, linecolor='white')

plt.xlabel('Метод пространстванной агрегации')
plt.ylabel('Метод агрегации аномалий')
plt.tight_layout()
plt.savefig('/kaggle/working/graphics/tmr.png', dpi=420, bbox_inches='tight')
plt.show()

In [ ]:
corr_matrix = shr.pivot(index='method', columns='space_method', values='Moran')
pvalue_matrix = shr.pivot(index='method', columns='space_method', values='P Moran')

mask = pvalue_matrix > 0.05

plt.figure(figsize=(10, 5))

sns.heatmap(
    corr_matrix, annot=True, fmt='.3f', cmap='bwr', vmin=-0.5, vmax=0.5, mask=mask, 
    cbar_kws={'label': 'Индекс Морана'}, linecolor='white'
)
sns.heatmap(corr_matrix, mask=~mask, cmap=['#f0f0f0'], annot=True, fmt='.3f', cbar=False, linecolor='white')

plt.xlabel('Метод пространстванной агрегации')
plt.ylabel('Метод агрегации аномалий')
plt.tight_layout()
plt.savefig('/kaggle/working/graphics/shr.png', dpi=420, bbox_inches='tight')
plt.show()

In [ ]:
corr_matrix = slr.pivot(index='method', columns='space_method', values='Moran')
pvalue_matrix = slr.pivot(index='method', columns='space_method', values='P Moran')

mask = pvalue_matrix > 0.05

plt.figure(figsize=(10, 5))

sns.heatmap(
    corr_matrix, annot=True, fmt='.3f', cmap='bwr', vmin=-0.5, vmax=0.5, mask=mask, 
    cbar_kws={'label': 'Индекс Морана'}, linecolor='white'
)
sns.heatmap(corr_matrix, mask=~mask, cmap=['#f0f0f0'], annot=True, fmt='.3f', cbar=False, linecolor='white')

plt.xlabel('Метод пространстванной агрегации')
plt.ylabel('Метод агрегации аномалий')
plt.tight_layout()
plt.savefig('/kaggle/working/graphics/slr.png', dpi=420, bbox_inches='tight')
plt.show()

### Spatial Models

In [ ]:
import numpy as np

In [ ]:
LMtests

In [ ]:
for filename in tqdm(total_list, total=len(total_list)):
    tmp = gpd.read_file(f'/kaggle/working/data/total/{filename}')
    tmp = tmp.dropna()
    name = filename[:-5]
    
    y = tmp['believers_pr']
    x = tmp[['anomaly']]
    w = Queen.from_dataframe(tmp, use_index=True, silence_warnings=True)
    w.transform = 'r'

    if report.loc[name, 'P Moran'] > 0.05:
        model = ML_Lag(y, x, w, slx_lags=0, spat_impacts='full')
        out = model.output
        
        report.loc[name, 'Coef A'] = out[out['var_names']=='anomaly']['coefficients'].iloc[0]
        report.loc[name, 'P Coef A'] = out[out['var_names']=='anomaly']['prob'].iloc[0]

        report.loc[name, 'Coef WA'] = np.nan
        report.loc[name, 'P Coef WA'] = np.nan

    else:
        model = ML_Lag(y, x, w, slx_lags=1, spat_impacts='full')
        out = model.output
        
        report.loc[name, 'Coef A'] = out[out['var_names']=='anomaly']['coefficients'].iloc[0]
        report.loc[name, 'P Coef A'] = out[out['var_names']=='anomaly']['prob'].iloc[0]
        
        report.loc[name, 'Coef WA'] = out[out['var_names']=='W_anomaly']['coefficients'].iloc[0]
        report.loc[name, 'P Coef WA'] = out[out['var_names']=='W_anomaly']['prob'].iloc[0]

In [ ]:
report.info()

In [ ]:
report['Coef A'] = report['Coef A'].astype(float)
report['P Coef A'] = report['P Coef A'].astype(float)

In [ ]:
report.info()

In [ ]:
report = report.sort_index()

shr = report.iloc[:12].copy().reset_index()
slr = report.iloc[12:24].copy().reset_index()
tmr = report.iloc[24:].copy().reset_index()

shr[['method', 'space_method']] = shr['index'].str.split('_', expand=True, n=3).iloc[:, -2:]
slr[['method', 'space_method']] = slr['index'].str.split('_', expand=True, n=3).iloc[:, -2:]
tmr[['method', 'space_method']] = tmr['index'].str.split('_', expand=True, n=3).iloc[:, -2:]

shr['method'] = shr['method'].map(method)
shr['space_method'] = shr['space_method'].map(space_method)
slr['method'] = slr['method'].map(method)
slr['space_method'] = slr['space_method'].map(space_method)
tmr['method'] = tmr['method'].map(method)
tmr['space_method'] = tmr['space_method'].map(space_method)

In [ ]:
cmap = plt.cm.bwr.copy()
cmap.set_bad('#f0f0f0')

In [ ]:
corr_matrix = tmr.pivot(index='method', columns='space_method', values='Coef A')
pvalue_matrix = tmr.pivot(index='method', columns='space_method', values='P Coef A')

mask = pvalue_matrix > 0.05

plt.figure(figsize=(10, 5))

sns.heatmap(
    corr_matrix, annot=True, fmt='.3f', cmap='bwr', vmin=-0.5, vmax=0.5, mask=mask, cbar_kws={'label': 'Коэффициент'}
)
sns.heatmap(corr_matrix, mask=~mask, cmap=['#f0f0f0'], annot=True, fmt='.3f', cbar=False)

plt.xlabel('Метод пространстванной агрегации')
plt.ylabel('Метод агрегации аномалий')
plt.tight_layout()
plt.savefig('/kaggle/working/graphics/tmr-a.png', dpi=420, bbox_inches='tight')
plt.show()

In [ ]:
corr_matrix = tmr.pivot(index='method', columns='space_method', values='Coef WA')
pvalue_matrix = tmr.pivot(index='method', columns='space_method', values='P Coef WA')

mask = pvalue_matrix > 0.05

plt.figure(figsize=(10, 5))

sns.heatmap(
    corr_matrix, annot=True, fmt='.3f', cmap=cmap, vmin=-0.5, vmax=0.5, mask=mask, cbar_kws={'label': 'Коэффициент'}
)
sns.heatmap(corr_matrix, mask=~mask, cmap=['#f0f0f0'], annot=True, fmt='.3f', cbar=False)
sns.heatmap(corr_matrix, mask=corr_matrix.notna(), cmap=['#f0f0f0'], annot=True, fmt='.3f', cbar=False)

plt.xlabel('Метод пространстванной агрегации')
plt.ylabel('Метод агрегации аномалий')
plt.tight_layout()
plt.savefig('/kaggle/working/graphics/tmr-wa.png', dpi=420, bbox_inches='tight')
plt.show()

In [ ]:
corr_matrix = shr.pivot(index='method', columns='space_method', values='Coef A')
pvalue_matrix = shr.pivot(index='method', columns='space_method', values='P Coef A')

mask = pvalue_matrix > 0.05

plt.figure(figsize=(10, 5))

sns.heatmap(
    corr_matrix, annot=True, fmt='.3f', cmap='bwr', vmin=-0.5, vmax=0.5, mask=mask, cbar_kws={'label': 'Коэффициент'}
)
sns.heatmap(corr_matrix, mask=~mask, cmap=['#f0f0f0'], annot=True, fmt='.3f', cbar=False)

plt.xlabel('Метод пространстванной агрегации')
plt.ylabel('Метод агрегации аномалий')
plt.tight_layout()
plt.savefig('/kaggle/working/graphics/shr-a.png', dpi=420, bbox_inches='tight')
plt.show()

In [ ]:
corr_matrix = shr.pivot(index='method', columns='space_method', values='Coef WA')
pvalue_matrix = shr.pivot(index='method', columns='space_method', values='P Coef WA')

mask = pvalue_matrix > 0.05

plt.figure(figsize=(10, 5))

sns.heatmap(
    corr_matrix, annot=True, fmt='.3f', cmap=cmap, vmin=-0.5, vmax=0.5, mask=mask, cbar_kws={'label': 'Коэффициент'}
)
sns.heatmap(corr_matrix, mask=~mask, cmap=['#f0f0f0'], annot=True, fmt='.3f', cbar=False)
sns.heatmap(corr_matrix, mask=corr_matrix.notna(), cmap=['#f0f0f0'], annot=True, fmt='.3f', cbar=False)

plt.xlabel('Метод пространстванной агрегации')
plt.ylabel('Метод агрегации аномалий')
plt.tight_layout()
plt.savefig('/kaggle/working/graphics/shr-wa.png', dpi=420, bbox_inches='tight')
plt.show()

In [ ]:
corr_matrix = slr.pivot(index='method', columns='space_method', values='Coef A')
pvalue_matrix = slr.pivot(index='method', columns='space_method', values='P Coef A')

mask = pvalue_matrix > 0.05

plt.figure(figsize=(10, 5))

sns.heatmap(
    corr_matrix, annot=True, fmt='.3f', cmap='bwr', vmin=-0.5, vmax=0.5, mask=mask, cbar_kws={'label': 'Коэффициент'}
)
sns.heatmap(corr_matrix, mask=~mask, cmap=['#f0f0f0'], annot=True, fmt='.3f', cbar=False)

plt.xlabel('Метод пространстванной агрегации')
plt.ylabel('Метод агрегации аномалий')
plt.tight_layout()
plt.savefig('/kaggle/working/graphics/slr-a.png', dpi=420, bbox_inches='tight')
plt.show()

In [ ]:
corr_matrix = slr.pivot(index='method', columns='space_method', values='Coef WA')
pvalue_matrix = slr.pivot(index='method', columns='space_method', values='P Coef WA')

mask = pvalue_matrix > 0.05

plt.figure(figsize=(10, 5))

sns.heatmap(
    corr_matrix, annot=True, fmt='.3f', cmap=cmap, vmin=-0.5, vmax=0.5, mask=mask, cbar_kws={'label': 'Коэффициент'}
)
sns.heatmap(corr_matrix, mask=~mask, cmap=['#f0f0f0'], annot=True, fmt='.3f', cbar=False)
sns.heatmap(corr_matrix, mask=corr_matrix.notna(), cmap=['#f0f0f0'], annot=True, fmt='.3f', cbar=False)

plt.xlabel('Метод пространстванной агрегации')
plt.ylabel('Метод агрегации аномалий')
plt.tight_layout()
plt.savefig('/kaggle/working/graphics/slr-wa.png', dpi=420, bbox_inches='tight')
plt.show()

In [ ]:
report.to_csv('/kaggle/working/report.csv')

### Spatiotemporal Analysis

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import geopandas as gpd
import contextily as ctx
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe

from tqdm.auto import tqdm
from matplotlib.colors import ListedColormap
from matplotlib.colors import to_rgba
from matplotlib.ticker import FuncFormatter
from matplotlib.patches import Patch
from shapely.affinity import translate

from esda.moran import Moran, Moran_Local, Moran_BV, Moran_Local_BV
from libpysal.weights.contiguity import Queen
from scipy.stats import kendalltau

In [ ]:
states = gpd.read_file('/kaggle/working/data/states.gpkg')
counties = gpd.read_file('/kaggle/working/data/counties.gpkg')
cells = gpd.read_file('/kaggle/working/data/cells.gpkg')

In [ ]:
counties['county'] = counties['county'].astype('int64')
cells['cell'] = cells['cell'].astype('int64')

In [ ]:
annual_list = os.listdir('/kaggle/working/data/annual/')

In [ ]:
os.makedirs('/kaggle/working/graphics/spatiotemporal/', exist_ok=True)

In [ ]:
def hsa(tmp, spatial_agg):
    final = pd.DataFrame()
    for i in tmp[spatial_agg].unique():
        tmp_i = tmp[tmp[spatial_agg]==i].copy()
        if len(tmp_i) != 10:
            continue
            
        l = len(tmp_i)
        qs = tmp_i['Q'].values
        ps = tmp_i['P'].values
        ms = tmp_i['M'].values
    
        qs = [q if p <= 0.05 else np.nan for q, p in zip(qs, ps)]
    
        def consecutive(lst, target):
            current_count = 0
            max_count = 0
            for element in lst:
                if element == target:
                    current_count += 1
                    max_count = max(max_count, current_count)
                else:
                    current_count = 0
            return max_count
    
        tau, tau_p = kendalltau(np.arange(len(ms)), ms)

        if (qs[-1] == 1) and (1 not in qs[:-1]):
            final.loc[i, 'type'] = 'Новая горячая точка'
        elif (consecutive(qs[6:], 1) > 1) and (1 not in qs[:6]) and ((qs.count(1) / l) < 0.9):
            final.loc[i, 'type'] = 'Последовательная горячая точка'
        elif (qs[-1] == 1) and ((qs.count(1) / l) >= 0.9) and (tau > 0) and (tau_p < 0.05):
            final.loc[i, 'type'] = 'Возрастающая горячая точка'
        elif ((qs.count(1) / l) >= 0.9) and ((tau_p > 0.05) or (tau == 0)):
            final.loc[i, 'type'] = 'Постоянная горячая точка'
        elif (qs[-1] == 1) and ((qs.count(1) / l) >= 0.9) and (tau < 0) and (tau_p < 0.05):
            final.loc[i, 'type'] = 'Убывающая горячая точка'
        elif (qs[-1] == 1) and (1 in qs[:-1]) and ((qs.count(1) / l) < 0.9) and (3 not in qs):
            final.loc[i, 'type'] = 'Спорадическая горячая точка'
        elif (qs[-1] == 1) and ((qs.count(1) / l) < 0.9) and (3 in qs):
            final.loc[i, 'type'] = 'Колеблющаяся горячая точка'
        elif (qs[-1] != 1) and ((qs.count(1) / l) >= 0.9):
            final.loc[i, 'type'] = 'Историческая горячая точка'
        elif (qs[-1] == 3) and (3 not in qs[:-1]):
            final.loc[i, 'type'] = 'Новая холодная точка'
        elif (consecutive(qs[6:], 3) > 1) and (3 not in qs[:6]) and ((qs.count(3) / l) < 0.9):
            final.loc[i, 'type'] = 'Последовательная холодная точка'
        elif (qs[-1] == 3) and ((qs.count(3) / l) >= 0.9) and (tau > 0) and (tau_p < 0.05):
            final.loc[i, 'type'] = 'Возрастающая холодная точка'
        elif ((qs.count(3) / l) >= 0.9) and ((tau_p > 0.05) or (tau == 0)):
            final.loc[i, 'type'] = 'Постоянная холодная точка'
        elif (qs[-1] == 3) and ((qs.count(3) / l) >= 0.9) and (tau < 0) and (tau_p < 0.05):
            final.loc[i, 'type'] = 'Убывающая холодная точка'
        elif (qs[-1] == 3) and (3 in qs[:-1]) and ((qs.count(3) / l) < 0.9) and (1 not in qs):
            final.loc[i, 'type'] = 'Спорадическая холодная точка'
        elif (qs[-1] == 3) and ((qs.count(3) / l) < 0.9) and (1 in qs):
            final.loc[i, 'type'] = 'Колеблющаяся холодная точка'
        elif (qs[-1] != 3) and ((qs.count(3) / l) >= 0.9):
            final.loc[i, 'type'] = 'Историческая холодная точка'
        else:
            final.loc[i, 'type'] = 'Закономерность не обнаружена'
    
    final = final.reset_index(names=[spatial_agg])
    if 'state' == spatial_agg:
        final = final.merge(states, on='state')
    elif 'county' == spatial_agg:
        final = final.merge(counties, on='county')
    else:
        final = final.merge(cells, on='cell')
    
    final = gpd.GeoDataFrame(final, geometry=final['geometry'], crs='EPSG:4326')
    return final

In [ ]:
def plot_STCHSA(final, name):
    color_dict = {
        'Новая горячая точка': 'pink',
        'Последовательная горячая точка': 'red',
        'Возрастающая горячая точка': 'firebrick',
        'Постоянная горячая точка': 'sienna',
        'Убывающая горячая точка': 'lightcoral',
        'Спорадическая горячая точка': 'orange',
        'Колеблющаяся горячая точка': 'gold',
        'Историческая горячая точка': 'bisque',
        'Новая холодная точка': 'paleturquoise',
        'Последовательная холодная точка': 'blue',
        'Возрастающая холодная точка': 'navy',
        'Постоянная холодная точка': 'indigo',
        'Убывающая холодная точка': 'cornflowerblue',
        'Спорадическая холодная точка': 'blueviolet',
        'Колеблющаяся холодная точка': 'orchid',
        'Историческая холодная точка': 'plum',
        'Закономерность не обнаружена': 'silver'
    }
    
    ordered_categories = list(color_dict.keys())
    final['type'] = pd.Categorical(final['type'], categories=ordered_categories)

    cmap = ListedColormap([color_dict[cat] for cat in ordered_categories])
    
    tmp_shadow = final.copy()
    dx, dy = 0.4, -0.4
    tmp_shadow.geometry = tmp_shadow.geometry.apply(
        lambda geom: translate(geom, xoff=dx, yoff=dy)
    )
    
    fig, ax = plt.subplots(figsize=(23, 9))
    
    tmp_shadow.plot(
        ax=ax,
        color=to_rgba('black', alpha=0.2),
        edgecolor='none',
        zorder=1,
        path_effects=[
            pe.withStroke(linewidth=6, foreground='gray', alpha=0.3),
            pe.Normal()
        ]
    )
    
    final.plot(
        'type', 
        legend=True, 
        categorical=True,
        ax=ax,
        cmap=cmap,
        edgecolor='white',
        linewidth=1,
    )
    
    def format_lon(x):
        return f"{abs(x):.0f}°{'W' if x < 0 else 'E'}"
    
    def format_lat(y):
        return f"{abs(y):.0f}°{'S' if y < 0 else 'N'}"

    ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: format_lon(x)))
    ax.yaxis.set_major_formatter(FuncFormatter(lambda y, _: format_lat(y)))
    ax.tick_params(axis='both', labelsize=14)
    
    ax.grid(color='black', linewidth=0.5)
    ctx.add_basemap(ax, source=ctx.providers.CartoDB.PositronNoLabels, crs=tmp.crs)
    ctx.add_basemap(ax, source=ctx.providers.CartoDB.PositronOnlyLabels, zoom=5, crs=tmp.crs)
    
    existing_categories = final['type'].unique()
    legend_elements = []
    for category in ordered_categories:
        if category in existing_categories:
            color = color_dict[category]
            alpha = 1
            edgecolor = 'black'
        else:
            color = '#CCCCCC'
            alpha = 0.3
            edgecolor = '#666666'
        
        legend_elements.append(Patch(facecolor=color, edgecolor=edgecolor, label=category, alpha=alpha))

    legend = ax.legend(
        handles=legend_elements,
        loc='center right',
        bbox_to_anchor=(1.45, 0.5),
        prop={'size': 16},
        frameon=False
    )
    
    for text, category in zip(legend.get_texts(), ordered_categories):
        if category not in existing_categories:
            text.set_color('#666666')
            text.set_alpha(0.7) 

    plt.savefig('/kaggle/working/graphics/spatiotemporal/'+name+'.png', dpi=300, bbox_inches='tight')
    plt.close()

In [ ]:
for filename in tqdm(annual_list, total=len(annual_list)):
    tmp = pd.read_csv(f'/kaggle/working/data/annual/{filename}')
    tmp = tmp.iloc[:, 1:]
    name = filename[:-4]

    if 'state' in name:
        tmp = tmp.merge(states, on='state')
        spatial_agg = 'state'
    elif 'county' in name:
        continue
    else:
        tmp = tmp.merge(cells, on='cell')
        spatial_agg = 'cell'

    tmp = gpd.GeoDataFrame(tmp, geometry=tmp['geometry'], crs='EPSG:4326')
    tmp = tmp[tmp['time'] != '2007-12-31']
    tmp = tmp[tmp['time'] != '2008-12-31']
    tmp = tmp[tmp['time'] != '2009-12-31']
    
    for time in tmp['time'].unique():
        print(time, end='\r')
        tmp_time = tmp[tmp['time']==time].copy()

        y = tmp_time['believers_pr'].values
        x = tmp_time['anomaly'].values
        w = Queen.from_dataframe(tmp_time, use_index=False, silence_warnings=True)
        w.transform = 'r'

        with warnings.catch_warnings():
            warnings.simplefilter('ignore', category=RuntimeWarning)
            moran_loc_bv = Moran_Local_BV(y, x, w, island_weight=0)

        tmp.loc[tmp['time']==time, 'Q'] = moran_loc_bv.q
        tmp.loc[tmp['time']==time, 'P'] = moran_loc_bv.p_sim
        tmp.loc[tmp['time']==time, 'M'] = moran_loc_bv.Is
    
    final = hsa(tmp, spatial_agg)
    plot_STCHSA(final, name)